In [ ]:
pip install adjustText

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import binned_statistic_2d
from adjustText import adjust_text
from IPython.display import display, HTML

# Exploring the Batters Situational ABS Splits Dataset

In [ ]:
# Import the dataset
batter_ABS_splits = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Batters.csv", encoding="latin1")

# Filter dataset to only include batters with at least 100 challenge opportunities
batter_ABS_splits_filtered = batter_ABS_splits[batter_ABS_splits["Total_Opp_Ovr"] >= 100]
batter_ABS_splits_filtered.describe()

In [ ]:
# =============================================================================
# DATA VALIDATION: Batters Situational Dataset
# =============================================================================

print("--- STARTING BATTERS DATA VALIDATION ---")
validation_passed = True

# Boundary Constraints: Ensure all rate/percentage columns are strictly between 0 and 1
rate_cols_batters = [col for col in batter_ABS_splits_filtered.columns if "Rate" in col or "rate" in col]
for col in rate_cols_batters:
    # Drop NaNs for validation since some fields might be empty if opportunities equal 0
    invalid_rows = batter_ABS_splits_filtered[(batter_ABS_splits_filtered[col] < 0) | (batter_ABS_splits_filtered[col] > 1)]
    if not invalid_rows.empty:
        print(f"❌ Boundary Alert: Column '{col}' contains {len(invalid_rows)} values outside [0, 1] bounds!")
        validation_passed = False

# Logical Subset Summations: Verify Home + Away equals Total Challenges
home_away_diff = (batter_ABS_splits_filtered["Home_Made"] + batter_ABS_splits_filtered["Away_Made"]) - batter_ABS_splits_filtered["Total_ChallengesMade"]
if not (home_away_diff == 0).all():
    mismatches = batter_ABS_splits_filtered[home_away_diff != 0]
    print(f"❌ Summation Mismatch: Home_Made + Away_Made does not equal Total_ChallengesMade for {len(mismatches)} players.")
    validation_passed = False

# Logical Subset Summations: Verify Pitch Type Groupings equal Total Challenges
# (FastBall_Made + BreakBall_Made + OtherPitch_Made = Total_ChallengesMade)
pitch_type_diff = (batter_ABS_splits_filtered["FastBall_Made"] +
                    batter_ABS_splits_filtered["BreakBall_Made"] +
                    batter_ABS_splits_filtered["OtherPitch_Made"]) - batter_ABS_splits_filtered["Total_ChallengesMade"]
if not (pitch_type_diff == 0).all():
    mismatches = batter_ABS_splits_filtered[pitch_type_diff != 0]
    print(f"❌ Summation Mismatch: Pitch groups (Fast/Break/Other) do not total Total_ChallengesMade for {len(mismatches)} rows.")
    validation_passed = False

# Check for Nulls/Missing Values in required core tracking fields
core_fields = ["batterId", "Total_Opp_Ovr", "Total_ChallengesMade", "Total_ChallengesOverturned"]
null_counts = batter_ABS_splits_filtered[core_fields].isnull().sum()
if null_counts.sum() > 0:
    print(f"❌ Missing Values found in core columns:\n{null_counts[null_counts > 0]}")
    validation_passed = False

if validation_passed:
    print("✅ Success: All Batter dataset structural validation checks passed successfully!")
else:
    print("⚠️ Warning: Vulnerabilities or errors detected in Batter structural integrity.")

## Box plots of the challenge and overturn rates among batters

In [ ]:
# Create a figure with 1 row and 2 columns for separate plots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Total_ChallengeRate on the first subplot
sns.boxplot(
    y="Total_ChallengeRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0],
    color="skyblue",
)
axes[0].set_title("Boxplot of Total Challenge Rate, Batters")
axes[0].set_ylabel("Challenge Rate")

# Plot Total_OverturnedRate on the second subplot
sns.boxplot(
    y="Total_OverturnedRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1],
    color="salmon",
)
axes[1].set_title("Boxplot of Total Overturned Rate, Batters")
axes[1].set_ylabel("Overturned Rate")

plt.tight_layout()
plt.show()

In [ ]:
# Code to identify the outliers
metrics = ["Total_ChallengeRate", "Total_OverturnedRate"]

for metric in metrics:
    q1 = batter_ABS_splits_filtered[metric].quantile(0.25)
    q3 = batter_ABS_splits_filtered[metric].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Isolate outlier rows
    outliers_df = batter_ABS_splits_filtered[(batter_ABS_splits_filtered[metric] < lower_bound) | (batter_ABS_splits_filtered[metric] > upper_bound)]

    print(f"\n🔹 Metric: {metric}")
    print(f"   IQR: {iqr:.4f} | Lower Bound: {lower_bound:.4f} | Upper Bound: {upper_bound:.4f}")

    if not outliers_df.empty:
        print(f"   Found {len(outliers_df)} statistical outlier(s):")
        # Display core columns for context
        display_cols = [
            "batterId",
            "firstName", "lastName",
            "Total_Opp_Ovr",
            "Total_ChallengesMade",
            "Total_ChallengeRate",
            "Total_OverturnedRate",
        ]
        print(
            outliers_df[display_cols].to_string(
                index=False,
                formatters={
                    "Total_ChallengeRate": "{:.4f}".format,
                    "Total_OverturnedRate": "{:.4f}".format,
                },
            )
        )
    else:
        print("   ✅ No statistical outliers detected for this metric.")

## Violin plots of the challenge and overturn rates among batters

In [ ]:
# Create a figure with 1 row and 2 columns for separate plots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Total_ChallengeRate on the first subplot
sns.violinplot(
    y="Total_ChallengeRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0],
    color="skyblue",
    inner="quartile",
)
axes[0].set_title("Distribution of Total Challenge Rate, Batters")
axes[0].set_ylabel("Challenge Rate")

# Plot Total_OverturnedRate on the second subplot
sns.violinplot(
    y="Total_OverturnedRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1],
    color="salmon",
    inner="quartile",
)
axes[1].set_title("Distribution of Total Overturned Rate, Batters")
axes[1].set_ylabel("Overturned Rate")

plt.tight_layout()
plt.show()

## Scatterplot of Challenge Rate vs Overturned Rate, Batters

In [ ]:
# Create the scatter plot
plt.figure(figsize=(8, 6))
sns.regplot(
    x="Total_ChallengeRate",
    y="Total_OverturnedRate",
    data=batter_ABS_splits_filtered,
    scatter_kws={"alpha": 0.6, "color": "darkgreen"},
    line_kws={"color": "red"},
)

# Customize labels and title
plt.title("Total Overturned Rate vs. Total Challenge Rate, Batters")
plt.xlabel("Total Challenge Rate")
plt.ylabel("Total Overturned Rate")

plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Violin plots of challenge rates by inning group, Batters

In [ ]:
# Create a figure with 3 row and 4 columns for separate plots side-by-side
fig, axes = plt.subplots(3, 4, figsize=(14, 6))

# Plot innings 1-3, overall
sns.violinplot(
    y="Inn1_3_Ovr_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][0],
    color="skyblue",
    inner="quartile",
)
axes[0][0].set_title("Innings 1-3, Overall")
axes[0][0].set_ylabel("")

# Plot innings 4-6, overall
sns.violinplot(
    y="Inn4_6_Ovr_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][1],
    color="salmon",
    inner="quartile",
)
axes[0][1].set_title("Innings 4-6, Overall")
axes[0][1].set_ylabel("")

# Plot innings 7-9, overall
sns.violinplot(
    y="Inn7_9_Ovr_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][2],
    color="skyblue",
    inner="quartile",
)
axes[0][2].set_title("Innings 7-9, Overall")
axes[0][2].set_ylabel("")

# Plot extra innings, overall
sns.violinplot(
    y="Inn10_Ovr_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][3],
    color="salmon",
    inner="quartile",
)
axes[0][3].set_title("Extra Innings, Overall")
axes[0][3].set_ylabel("")

# Plot innings 1-3, both left
sns.violinplot(
    y="Inn1_3_Chg2_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][0],
    color="salmon",
    inner="quartile",
)
axes[1][0].set_title("Innings 1-3, Both Left")
axes[1][0].set_ylabel("")

# Plot innings 4-6, Both Left
sns.violinplot(
    y="Inn4_6_Chg2_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][1],
    color="skyblue",
    inner="quartile",
)
axes[1][1].set_title("Innings 4-6, Both Left")
axes[1][1].set_ylabel("")

# Plot innings 7-9, Both Left
sns.violinplot(
    y="Inn7_9_Chg2_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][2],
    color="salmon",
    inner="quartile",
)
axes[1][2].set_title("Innings 7-9, Both Left")
axes[1][2].set_ylabel("")

# Plot extra innings, Both Left
sns.violinplot(
    y="Inn10_Chg2_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][3],
    color="skyblue",
    inner="quartile",
)
axes[1][3].set_title("Extra Innings, Both Left")
axes[1][3].set_ylabel("")

# Plot innings 1-3, One left
sns.violinplot(
    y="Inn1_3_Chg1_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][0],
    color="skyblue",
    inner="quartile",
)
axes[2][0].set_title("Innings 1-3, One Left")
axes[2][0].set_ylabel("")

# Plot innings 4-6, One Left
sns.violinplot(
    y="Inn4_6_Chg1_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][1],
    color="salmon",
    inner="quartile",
)
axes[2][1].set_title("Innings 4-6, One Left")
axes[2][1].set_ylabel("")

# Plot innings 7-9, One Left
sns.violinplot(
    y="Inn7_9_Chg1_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][2],
    color="skyblue",
    inner="quartile",
)
axes[2][2].set_title("Innings 7-9, One Left")
axes[2][2].set_ylabel("")

# Plot extra innings, One Left
sns.violinplot(
    y="Inn10_Chg1_Rate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][3],
    color="salmon",
    inner="quartile",
)
axes[2][3].set_title("Extra Innings, One Left")
axes[2][3].set_ylabel("")

#plt.title("Violin Plots of Challenge Rates, Batters, By Inning Group and Challenges Remaining")
plt.tight_layout()
plt.show()

## Violin plots of overturn rates by inning group, Batters

In [ ]:
# Create a figure with 3 row and 4 columns for separate plots side-by-side
fig, axes = plt.subplots(3, 4, figsize=(14, 6))

# Plot innings 1-3, overall
sns.violinplot(
    y="Inn1_3_Ovr_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][0],
    color="skyblue",
    inner="quartile",
)
axes[0][0].set_title("Innings 1-3, Overall")
axes[0][0].set_ylabel("")

# Plot innings 4-6, overall
sns.violinplot(
    y="Inn4_6_Ovr_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][1],
    color="salmon",
    inner="quartile",
)
axes[0][1].set_title("Innings 4-6, Overall")
axes[0][1].set_ylabel("")

# Plot innings 7-9, overall
sns.violinplot(
    y="Inn7_9_Ovr_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][2],
    color="skyblue",
    inner="quartile",
)
axes[0][2].set_title("Innings 7-9, Overall")
axes[0][2].set_ylabel("")

# Plot extra innings, overall
sns.violinplot(
    y="Inn10_Ovr_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[0][3],
    color="salmon",
    inner="quartile",
)
axes[0][3].set_title("Extra Innings, Overall")
axes[0][3].set_ylabel("")

# Plot innings 1-3, both left
sns.violinplot(
    y="Inn1_3_Chg2_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][0],
    color="salmon",
    inner="quartile",
)
axes[1][0].set_title("Innings 1-3, Both Left")
axes[1][0].set_ylabel("")

# Plot innings 4-6, Both Left
sns.violinplot(
    y="Inn4_6_Chg2_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][1],
    color="skyblue",
    inner="quartile",
)
axes[1][1].set_title("Innings 4-6, Both Left")
axes[1][1].set_ylabel("")

# Plot innings 7-9, Both Left
sns.violinplot(
    y="Inn7_9_Chg2_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][2],
    color="salmon",
    inner="quartile",
)
axes[1][2].set_title("Innings 7-9, Both Left")
axes[1][2].set_ylabel("")

# Plot extra innings, Both Left
sns.violinplot(
    y="Inn10_Chg2_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[1][3],
    color="skyblue",
    inner="quartile",
)
axes[1][3].set_title("Extra Innings, Both Left")
axes[1][3].set_ylabel("")

# Plot innings 1-3, One left
sns.violinplot(
    y="Inn1_3_Chg1_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][0],
    color="skyblue",
    inner="quartile",
)
axes[2][0].set_title("Innings 1-3, One Left")
axes[2][0].set_ylabel("")

# Plot innings 4-6, One Left
sns.violinplot(
    y="Inn4_6_Chg1_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][1],
    color="salmon",
    inner="quartile",
)
axes[2][1].set_title("Innings 4-6, One Left")
axes[2][1].set_ylabel("")

# Plot innings 7-9, One Left
sns.violinplot(
    y="Inn7_9_Chg1_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][2],
    color="skyblue",
    inner="quartile",
)
axes[2][2].set_title("Innings 7-9, One Left")
axes[2][2].set_ylabel("")

# Plot extra innings, One Left
sns.violinplot(
    y="Inn10_Chg1_OvrRate",
    data=batter_ABS_splits_filtered,
    ax=axes[2][3],
    color="salmon",
    inner="quartile",
)
axes[2][3].set_title("Extra Innings, One Left")
axes[2][3].set_ylabel("")

plt.tight_layout()
plt.show()

## Stacked Bar Chart, Pitch Type Groupings for Batters

In [ ]:
# Calculate the totals for the full dataset
pitch_totals = batter_ABS_splits[
    ["FastBall_Made", "BreakBall_Made", "OtherPitch_Made"]
].sum()
pitch_overturned = batter_ABS_splits[
    ["FastBall_Over", "BreakBall_Over", "OtherPitch_Over"]
].sum()

# Create DataFrame for plotting
df_pitch = pd.DataFrame(
    {
        "Overturned": pitch_overturned.values,
        "Total Challenges": pitch_totals.values,
    },
    index=["Fastballs", "Breaking Balls", "Off-Speed Pitches"],
)

# Calculate Upheld challenges cleanly across the 3 rows
df_pitch["Upheld"] = df_pitch["Total Challenges"] - df_pitch["Overturned"]
# Plotting
fig, ax = plt.subplots(figsize=(8, 6))

# Plotting the stacked bar chart cleanly
df_pitch[["Overturned", "Upheld"]].plot(
    kind="bar", stacked=True, color=["coral", "teal"], ax=ax, edgecolor="black"
)

# Loop through the bars to display the numerical values on each section
for container in ax.containers:
    ax.bar_label(container, label_type="center", fontsize=10, weight="bold", color="white")

# Customizing the labels and titles for your report
ax.set_title(
    "ABS Challenges & Outcomes by Pitch Type (Batters)",
    fontsize=12,
    weight="bold",
)
ax.set_ylabel("Number of Actions", fontsize=10)

# Rotate the clean index labels to sit perfectly horizontal
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

df_pitch

# Exploring the Catchers Situational ABS Splits Dataset

In [ ]:
# Import the dataset
catcher_ABS_splits = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Catchers.csv", encoding="latin1")

# Filter dataset to only include batters with at least 100 challenge opportunities
catcher_ABS_splits_filtered = catcher_ABS_splits[catcher_ABS_splits["Total_Opp_Ovr"] >= 100]
catcher_ABS_splits_filtered.describe()

In [ ]:
# =============================================================================
# DATA VALIDATION: Catchers Situational Dataset
# =============================================================================

print("\n--- STARTING CATCHERS DATA VALIDATION ---")
catcher_validation_passed = True

# Boundary Constraints: Ensure all rate/percentage columns are strictly between 0 and 1
rate_cols_catchers = [col for col in catcher_ABS_splits_filtered.columns if "Rate" in col or "rate" in col]
for col in rate_cols_catchers:
    invalid_rows = catcher_ABS_splits_filtered[(catcher_ABS_splits_filtered[col] < 0) | (catcher_ABS_splits_filtered[col] > 1)]
    if not invalid_rows.empty:
        print(f"❌ Boundary Alert: Column '{col}' contains {len(invalid_rows)} values outside [0, 1] bounds!")
        catcher_validation_passed = False

# Logical Subset Summations: Verify Home + Away equals Total Challenges
catcher_home_away_diff = (catcher_ABS_splits_filtered["Home_Made"] + catcher_ABS_splits_filtered["Away_Made"]) - catcher_ABS_splits_filtered["Total_ChallengesMade"]
if not (catcher_home_away_diff == 0).all():
    mismatches = catcher_ABS_splits_filtered[catcher_home_away_diff != 0]
    print(f"❌ Summation Mismatch: Home_Made + Away_Made does not equal Total_ChallengesMade for {len(mismatches)} catchers.")
    catcher_validation_passed = False

# Logical Subset Summations: Verify Pitch Type Groupings equal Total Challenges
catcher_pitch_type_diff = (catcher_ABS_splits_filtered["FastBall_Made"] +
                           catcher_ABS_splits_filtered["BreakBall_Made"] +
                           catcher_ABS_splits_filtered["OtherPitch_Made"]) - catcher_ABS_splits_filtered["Total_ChallengesMade"]
if not (catcher_pitch_type_diff == 0).all():
    mismatches = catcher_ABS_splits_filtered[catcher_pitch_type_diff != 0]
    print(f"❌ Summation Mismatch: Pitch groups do not total Total_ChallengesMade for {len(mismatches)} catchers.")
    catcher_validation_passed = False

# Check for Nulls/Missing Values in required core tracking fields
catcher_core_fields = ["catcherId", "Total_Opp_Ovr", "Total_ChallengesMade", "Total_ChallengesOverturned"]
catcher_null_counts = catcher_ABS_splits_filtered[catcher_core_fields].isnull().sum()
if catcher_null_counts.sum() > 0:
    print(f"❌ Missing Values found in catcher core columns:\n{catcher_null_counts[catcher_null_counts > 0]}")
    catcher_validation_passed = False

if catcher_validation_passed:
    print("✅ Success: All Catcher dataset structural validation checks passed successfully!")
else:
    print("⚠️ Warning: Vulnerabilities or errors detected in Catcher structural integrity.")

## Box plots of the challenge and overturn rates among catchers

In [ ]:
# Create a figure with 1 row and 2 columns for separate plots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Total_ChallengeRate on the first subplot
sns.boxplot(
    y="Total_ChallengeRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0],
    color="skyblue",
)
axes[0].set_title("Boxplot of Total Challenge Rate, Catchers")
axes[0].set_ylabel("Challenge Rate")

# Plot Total_OverturnedRate on the second subplot
sns.boxplot(
    y="Total_OverturnedRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1],
    color="salmon",
)
axes[1].set_title("Boxplot of Total Overturned Rate, Catchers")
axes[1].set_ylabel("Overturned Rate")

plt.tight_layout()
plt.show()

In [ ]:
# Code to identify the outliers
metrics = ["Total_ChallengeRate", "Total_OverturnedRate"]

for metric in metrics:
    q1 = catcher_ABS_splits_filtered[metric].quantile(0.25)
    q3 = catcher_ABS_splits_filtered[metric].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Isolate outlier rows
    outliers_df = catcher_ABS_splits_filtered[(catcher_ABS_splits_filtered[metric] < lower_bound) | (catcher_ABS_splits_filtered[metric] > upper_bound)]

    print(f"\n🔹 Metric: {metric}")
    print(f"   IQR: {iqr:.4f} | Lower Bound: {lower_bound:.4f} | Upper Bound: {upper_bound:.4f}")

    if not outliers_df.empty:
        print(f"   Found {len(outliers_df)} statistical outlier(s):")
        # Display core columns for context
        display_cols = [
            "catcherId",
            "firstName", "lastName",
            "Total_Opp_Ovr",
            "Total_ChallengesMade",
            "Total_ChallengeRate",
            "Total_OverturnedRate",
        ]
        print(
            outliers_df[display_cols].to_string(
                index=False,
                formatters={
                    "Total_ChallengeRate": "{:.4f}".format,
                    "Total_OverturnedRate": "{:.4f}".format,
                },
            )
        )
    else:
        print("   ✅ No statistical outliers detected for this metric.")

## Violin plots of the challenge and overturn rates among catchers

In [ ]:
# Create a figure with 1 row and 2 columns for separate plots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Total_ChallengeRate on the first subplot
sns.violinplot(
    y="Total_ChallengeRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0],
    color="skyblue",
    inner="quartile",
)
axes[0].set_title("Distribution of Total Challenge Rate, Catchers")
axes[0].set_ylabel("Challenge Rate")

# Plot Total_OverturnedRate on the second subplot
sns.violinplot(
    y="Total_OverturnedRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1],
    color="salmon",
    inner="quartile",
)
axes[1].set_title("Distribution of Total Overturned Rate, Catchers")
axes[1].set_ylabel("Overturned Rate")

plt.tight_layout()
plt.show()

## Scatterplot of Challenge Rate vs Overturned Rate, Catchers

In [ ]:
# Create the scatter plot
plt.figure(figsize=(8, 6))
sns.regplot(
    x="Total_ChallengeRate",
    y="Total_OverturnedRate",
    data=catcher_ABS_splits_filtered,
    scatter_kws={"alpha": 0.6, "color": "darkgreen"},
    line_kws={"color": "red"},
)

# Customize labels and title
plt.title("Total Overturned Rate vs. Total Challenge Rate, Catchers")
plt.xlabel("Total Challenge Rate")
plt.ylabel("Total Overturned Rate")

plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Violin plots of challenge rates by inning group, Catcher

In [ ]:
# Create a figure with 3 row and 4 columns for separate plots side-by-side
fig, axes = plt.subplots(3, 4, figsize=(14, 6))

# Plot innings 1-3, overall
sns.violinplot(
    y="Inn1_3_Ovr_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][0],
    color="skyblue",
    inner="quartile",
)
axes[0][0].set_title("Innings 1-3, Overall")
axes[0][0].set_ylabel("")

# Plot innings 4-6, overall
sns.violinplot(
    y="Inn4_6_Ovr_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][1],
    color="salmon",
    inner="quartile",
)
axes[0][1].set_title("Innings 4-6, Overall")
axes[0][1].set_ylabel("")

# Plot innings 7-9, overall
sns.violinplot(
    y="Inn7_9_Ovr_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][2],
    color="skyblue",
    inner="quartile",
)
axes[0][2].set_title("Innings 7-9, Overall")
axes[0][2].set_ylabel("")

# Plot extra innings, overall
sns.violinplot(
    y="Inn10_Ovr_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][3],
    color="salmon",
    inner="quartile",
)
axes[0][3].set_title("Extra Innings, Overall")
axes[0][3].set_ylabel("")

# Plot innings 1-3, both left
sns.violinplot(
    y="Inn1_3_Chg2_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][0],
    color="salmon",
    inner="quartile",
)
axes[1][0].set_title("Innings 1-3, Both Left")
axes[1][0].set_ylabel("")

# Plot innings 4-6, Both Left
sns.violinplot(
    y="Inn4_6_Chg2_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][1],
    color="skyblue",
    inner="quartile",
)
axes[1][1].set_title("Innings 4-6, Both Left")
axes[1][1].set_ylabel("")

# Plot innings 7-9, Both Left
sns.violinplot(
    y="Inn7_9_Chg2_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][2],
    color="salmon",
    inner="quartile",
)
axes[1][2].set_title("Innings 7-9, Both Left")
axes[1][2].set_ylabel("")

# Plot extra innings, Both Left
sns.violinplot(
    y="Inn10_Chg2_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][3],
    color="skyblue",
    inner="quartile",
)
axes[1][3].set_title("Extra Innings, Both Left")
axes[1][3].set_ylabel("")

# Plot innings 1-3, One left
sns.violinplot(
    y="Inn1_3_Chg1_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][0],
    color="skyblue",
    inner="quartile",
)
axes[2][0].set_title("Innings 1-3, One Left")
axes[2][0].set_ylabel("")

# Plot innings 4-6, One Left
sns.violinplot(
    y="Inn4_6_Chg1_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][1],
    color="salmon",
    inner="quartile",
)
axes[2][1].set_title("Innings 4-6, One Left")
axes[2][1].set_ylabel("")

# Plot innings 7-9, One Left
sns.violinplot(
    y="Inn7_9_Chg1_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][2],
    color="skyblue",
    inner="quartile",
)
axes[2][2].set_title("Innings 7-9, One Left")
axes[2][2].set_ylabel("")

# Plot extra innings, One Left
sns.violinplot(
    y="Inn10_Chg1_Rate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][3],
    color="salmon",
    inner="quartile",
)
axes[2][3].set_title("Extra Innings, One Left")
axes[2][3].set_ylabel("")

#plt.title("Violin Plots of Challenge Rates, Catchers, By Inning Group and Challenges Remaining")
plt.tight_layout()
plt.show()

## Violin plots of overturn rates by inning group, Catcher

In [ ]:
# Create a figure with 3 row and 4 columns for separate plots side-by-side
fig, axes = plt.subplots(3, 4, figsize=(14, 6))

# Plot innings 1-3, overall
sns.violinplot(
    y="Inn1_3_Ovr_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][0],
    color="skyblue",
    inner="quartile",
)
axes[0][0].set_title("Innings 1-3, Overall")
axes[0][0].set_ylabel("")

# Plot innings 4-6, overall
sns.violinplot(
    y="Inn4_6_Ovr_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][1],
    color="salmon",
    inner="quartile",
)
axes[0][1].set_title("Innings 4-6, Overall")
axes[0][1].set_ylabel("")

# Plot innings 7-9, overall
sns.violinplot(
    y="Inn7_9_Ovr_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][2],
    color="skyblue",
    inner="quartile",
)
axes[0][2].set_title("Innings 7-9, Overall")
axes[0][2].set_ylabel("")

# Plot extra innings, overall
sns.violinplot(
    y="Inn10_Ovr_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[0][3],
    color="salmon",
    inner="quartile",
)
axes[0][3].set_title("Extra Innings, Overall")
axes[0][3].set_ylabel("")

# Plot innings 1-3, both left
sns.violinplot(
    y="Inn1_3_Chg2_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][0],
    color="salmon",
    inner="quartile",
)
axes[1][0].set_title("Innings 1-3, Both Left")
axes[1][0].set_ylabel("")

# Plot innings 4-6, Both Left
sns.violinplot(
    y="Inn4_6_Chg2_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][1],
    color="skyblue",
    inner="quartile",
)
axes[1][1].set_title("Innings 4-6, Both Left")
axes[1][1].set_ylabel("")

# Plot innings 7-9, Both Left
sns.violinplot(
    y="Inn7_9_Chg2_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][2],
    color="salmon",
    inner="quartile",
)
axes[1][2].set_title("Innings 7-9, Both Left")
axes[1][2].set_ylabel("")

# Plot extra innings, Both Left
sns.violinplot(
    y="Inn10_Chg2_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[1][3],
    color="skyblue",
    inner="quartile",
)
axes[1][3].set_title("Extra Innings, Both Left")
axes[1][3].set_ylabel("")

# Plot innings 1-3, One left
sns.violinplot(
    y="Inn1_3_Chg1_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][0],
    color="skyblue",
    inner="quartile",
)
axes[2][0].set_title("Innings 1-3, One Left")
axes[2][0].set_ylabel("")

# Plot innings 4-6, One Left
sns.violinplot(
    y="Inn4_6_Chg1_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][1],
    color="salmon",
    inner="quartile",
)
axes[2][1].set_title("Innings 4-6, One Left")
axes[2][1].set_ylabel("")

# Plot innings 7-9, One Left
sns.violinplot(
    y="Inn7_9_Chg1_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][2],
    color="skyblue",
    inner="quartile",
)
axes[2][2].set_title("Innings 7-9, One Left")
axes[2][2].set_ylabel("")

# Plot extra innings, One Left
sns.violinplot(
    y="Inn10_Chg1_OvrRate",
    data=catcher_ABS_splits_filtered,
    ax=axes[2][3],
    color="salmon",
    inner="quartile",
)
axes[2][3].set_title("Extra Innings, One Left")
axes[2][3].set_ylabel("")

plt.tight_layout()
plt.show()

## Home vs. Away Batter Challenge Behavior (Paired Violin Plot)

In [ ]:
plt.figure(figsize=(8, 6))
# Melts the dataframe to compare Home vs Away rates side-by-side
venue_df = batter_ABS_splits_filtered.melt(
    value_vars=["Home_Rate", "Away_Rate"],
    var_name="Venue",
    value_name="Challenge Rate",
)
venue_df["Venue"] = venue_df["Venue"].map(
    {"Home_Rate": "Home", "Away_Rate": "Away"}
)

sns.violinplot(x="Venue", y="Challenge Rate", data=venue_df, palette="Pastel1", inner="quartile")
plt.title("Batter Challenge Rates: Home vs. Away")
plt.ylabel("Challenge Rate Percentage")
plt.xlabel("Game Venue")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

## Stacked Bar Chart, Pitch Type Groupings for Catchers

In [ ]:
# Calculate the totals for the full dataset
pitch_totals = catcher_ABS_splits[
    ["FastBall_Made", "BreakBall_Made", "OtherPitch_Made"]
].sum()
pitch_overturned = catcher_ABS_splits[
    ["FastBall_Over", "BreakBall_Over", "OtherPitch_Over"]
].sum()

# Create DataFrame for plotting
df_pitch = pd.DataFrame(
    {
        "Overturned": pitch_overturned.values,
        "Total Challenges": pitch_totals.values,
    },
    index=["Fastballs", "Breaking Balls", "Off-Speed Pitches"],
)

# Calculate Upheld challenges cleanly across the 3 rows
df_pitch["Upheld"] = df_pitch["Total Challenges"] - df_pitch["Overturned"]
# Plotting
fig, ax = plt.subplots(figsize=(8, 6))

# Plotting the stacked bar chart cleanly
df_pitch[["Overturned", "Upheld"]].plot(
    kind="bar", stacked=True, color=["coral", "teal"], ax=ax, edgecolor="black"
)

# Loop through the bars to display the numerical values on each section
for container in ax.containers:
    ax.bar_label(container, label_type="center", fontsize=10, weight="bold", color="white")

# Customizing the labels and titles for your report
ax.set_title(
    "ABS Challenges & Outcomes by Pitch Type (Catchers)",
    fontsize=12,
    weight="bold",
)
ax.set_ylabel("Number of Actions", fontsize=10)

# Rotate the clean index labels to sit perfectly horizontal
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

df_pitch

## Catcher Framework Performance Distribution (Density Plot)

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(
    catcher_ABS_splits_filtered["Total_OverturnedRate"],
    shade=True,
    color="olive",
    bw_adjust=0.5,
)
plt.title("Density Distribution of Catcher Overturn Rates")
plt.xlabel("Total Overturned Rate (Accuracy)")
plt.ylabel("Density")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Catchers' Workload vs. Challenge Frequency (Scatterplot)

In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(
    x="Total_Opp_Ovr",
    y="Total_ChallengeRate",
    data=catcher_ABS_splits_filtered,
    scatter_kws={"alpha": 0.6, "color": "darkgreen"},
    line_kws={"color": "red"},
)
plt.title("Catcher Workload vs. Challenge Rate Frequency")
plt.xlabel("Total Challenge Opportunities")
plt.ylabel("Challenge Rate")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Home vs. Away Batter Challenge Behavior (Paired Violin Plot)

In [ ]:
plt.figure(figsize=(8, 6))
# Melts the dataframe to compare Home vs Away rates side-by-side
venue_df = catcher_ABS_splits_filtered.melt(
    value_vars=["Home_Rate", "Away_Rate"],
    var_name="Venue",
    value_name="Challenge Rate",
)
venue_df["Venue"] = venue_df["Venue"].map(
    {"Home_Rate": "Home", "Away_Rate": "Away"}
)

sns.violinplot(x="Venue", y="Challenge Rate", data=venue_df, palette="Pastel1", inner="quartile")
plt.title("Catcher Challenge Rates: Home vs. Away")
plt.ylabel("Challenge Rate Percentage")
plt.xlabel("Game Venue")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

# Exploring the Intersection of the Catchers and Batters Situational ABS Splits Dataset

In [ ]:
# =============================================================================
# JOINING DATASETS & GENERATING SCATTERPLOTS (Players in Both Datasets)
# =============================================================================

# Merge the filtered datasets on the player ID
# (batterId in the batter dataset matches catcherId in the catcher dataset)
merged_splits = pd.merge(
    batter_ABS_splits_filtered,
    catcher_ABS_splits_filtered,
    left_on="batterId",
    right_on="catcherId",
    suffixes=("_batter", "_catcher"),
)

print(
    f"📊 Found {len(merged_splits)} players who qualify in both filtered datasets.\n"
)

if not merged_splits.empty:
    # Set up a side-by-side plotting area
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Scatterplot 1: Challenge Rates
    sns.scatterplot(
        x="Total_ChallengeRate_batter",
        y="Total_ChallengeRate_catcher",
        data=merged_splits,
        ax=axes[0],
        color="teal",
        s=100,
        alpha=0.8,
    )
    axes[0].set_title("Challenge Rate: As Batter vs. As Catcher")
    axes[0].set_xlabel("Total Challenge Rate (As Batter)")
    axes[0].set_ylabel("Total Challenge Rate (As Catcher)")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    # Add identity line (x=y) to show if they challenge more as a batter or catcher
    max_val_chg = max(
        merged_splits["Total_ChallengeRate_batter"].max(),
        merged_splits["Total_ChallengeRate_catcher"].max(),
    )
    axes[0].plot([0, max_val_chg], [0, max_val_chg], "r--", alpha=0.5, label="1:1 Equality")
    axes[0].legend()

    # Scatterplot 2: Overturned Rates
    sns.scatterplot(
        x="Total_OverturnedRate_batter",
        y="Total_OverturnedRate_catcher",
        data=merged_splits,
        ax=axes[1],
        color="coral",
        s=100,
        alpha=0.8,
    )
    axes[1].set_title("Overturn Rate: As Batter vs. As Catcher")
    axes[1].set_xlabel("Total Overturned Rate (As Batter)")
    axes[1].set_ylabel("Total Overturned Rate (As Catcher)")
    axes[1].grid(True, linestyle="--", alpha=0.5)

    # Add identity line (x=y)
    axes[1].plot([0, 1], [0, 1], "r--", alpha=0.5, label="1:1 Equality")
    axes[1].legend()

    plt.suptitle(
        "ABS Matrix Comparison for Catchers (Minimum 100 Opps per Split)",
        fontsize=14,
        weight="bold",
    )
    plt.tight_layout()
    plt.show()

    # Display the matched players for reference
    print("📋 Matched Players Profile:")
    display_cols = [
        "batterId",
        "firstName_batter",
        "lastName_batter",
        "Total_ChallengeRate_batter",
        "Total_ChallengeRate_catcher",
        "Total_OverturnedRate_batter",
        "Total_OverturnedRate_catcher",
    ]
    print(merged_splits[display_cols].to_string(index=False))
else:
    print(
        "⚠️ No players matched both filters. Consider lowering the 'Total_Opp_Ovr' threshold."
    )


# Exploring the Challengeable Pitches Dataset

## Various heat maps of challenge rate based on zone location

In [ ]:
# Import the datasets
bat_challPitches = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Batters.csv", encoding="latin1")
def_challPitches = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Defense.csv", encoding="latin1")
bat_challPitches.head()

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
bat_challPitches["plateTimeY_topDiff"] = (
    bat_challPitches["plateTimeY"] - bat_challPitches["szTop"]
)
bat_challPitches["plateTimeY_BotDiff"] = (
    bat_challPitches["plateTimeY"] - bat_challPitches["szBottom"]
)
def_challPitches["plateTimeY_topDiff"] = (
    def_challPitches["plateTimeY"] - def_challPitches["szTop"]
)
def_challPitches["plateTimeY_BotDiff"] = (
    def_challPitches["plateTimeY"] - def_challPitches["szBottom"]
)

In [ ]:
# =============================================================================
# SPATIAL ANALYSIS: CHALLENGE RATES RELATIVE TO ZONE BOUNDARIES
# =============================================================================

# Define spatial binning grids to aggregate the challenge rates
# Adjust bin counts or ranges based on your data distribution if needed
x_bins = pd.cut(bat_challPitches["plateTimeX"], bins=15)

# Bin the vertical differences
y_top_bins = pd.cut(bat_challPitches["plateTimeY_topDiff"], bins=15)
y_bot_bins = pd.cut(bat_challPitches["plateTimeY_BotDiff"], bins=15)

# Create Pivot Tables calculating the mean of "hasABSChallenge" (Challenge Rate)
heatmap_top = bat_challPitches.pivot_table(
    index=y_top_bins,
    columns=x_bins,
    values="hasABSChallenge",
    aggfunc="mean",
    observed=False,
)

heatmap_bot = bat_challPitches.pivot_table(
    index=y_bot_bins,
    columns=x_bins,
    values="hasABSChallenge",
    aggfunc="mean",
    observed=False,
)

# Invert Y-axis index format so higher pitches appear at the top of the heatmap visual
heatmap_top = heatmap_top.iloc[::-1]
heatmap_bot = heatmap_bot.iloc[::-1]

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap 1: Top of the Strike Zone Boundary Difference
sns.heatmap(
    heatmap_top,
    ax=axes[0],
    cmap="viridis",
    cbar_kws={"label": "Challenge Rate (Mean of hasABSChallenge)"},
)
axes[0].set_title("Challenge Rate Relative to the Top of the Strike Zone")
axes[0].set_ylabel("Vertical Distance from Top of Zone (plateTimeY_topDiff)")
axes[0].set_xlabel("Horizontal Plate Coordinate (plateTimeX)")

# Heatmap 2: Bottom of the Strike Zone Boundary Difference
sns.heatmap(
    heatmap_bot,
    ax=axes[1],
    cmap="viridis",
    cbar_kws={"label": "Challenge Rate (Mean of hasABSChallenge)"},
)
axes[1].set_title("Challenge Rate Relative to the Bottom of the Strike Zone")
axes[1].set_ylabel("Vertical Distance from Bottom of Zone (plateTimeY_BotDiff)")
axes[1].set_xlabel("Horizontal Plate Coordinate (plateTimeX)")

# Format label visibility across the large matrix grids
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle(
    "Spatial Analysis of ABS Challenge Probability Relative to Strike Zone Horizons, Batters",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SPATIAL ANALYSIS: TWO-DIMENSIONAL DISTRIBUTION WITH ZONE BOUNDARIES
# =============================================================================

# Isolate only rows where a challenge actually occurred
challenged_pitches = bat_challPitches[bat_challPitches["hasABSChallenge"] == 1]

# Explicitly drop any missing/NaN tracking records to prevent Numpy range errors
plot_data = challenged_pitches.dropna(
    subset=["plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff"]
)

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Top of the Strike Zone Horizon ---
counts1, xedges1, yedges1, im1 = axes[0].hist2d(
    plot_data["plateTimeX"],
    plot_data["plateTimeY_topDiff"],
    bins=25,
    cmap="viridis",
    cmin=1,  # Hides empty bins for visual clarity
)
fig.colorbar(im1, ax=axes[0], label="Volume of Challenges")

# Overlay theoretical boundaries (Home plate edges and the Y=0 boundary line)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=2, label="Top of Zone")
axes[0].axvline(
    x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges"
)
axes[0].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[0].set_title("Challenge Density Relative to the Upper Zone Boundary")
axes[0].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[0].set_ylabel("Vertical Distance from Top Edge (Feet)")
axes[0].legend(loc="upper right")
axes[0].set_xlim(-2.0, 2.0)  # Zoomed in closely around the plate area

# --- Plot 2: Bottom of the Strike Zone Horizon ---
counts2, xedges2, yedges2, im2 = axes[1].hist2d(
    plot_data["plateTimeX"],
    plot_data["plateTimeY_BotDiff"],  # Fixed a variable name typo here as well
    bins=25,
    cmap="viridis",
    cmin=1,
)
fig.colorbar(im2, ax=axes[1], label="Volume of Challenges")

# Overlay theoretical boundaries
axes[1].axhline(
    y=0, color="red", linestyle="--", linewidth=2, label="Bottom of Zone"
)
axes[1].axvline(
    x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges"
)
axes[1].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[1].set_title("Challenge Density Relative to the Lower Zone Boundary")
axes[1].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[1].set_ylabel("Vertical Distance from Bottom Edge (Feet)")
axes[1].legend(loc="upper right")
axes[1].set_xlim(-2.0, 2.0)

plt.suptitle(
    "Spatial Map of ABS Challenge Allocations Against Physical Strike Zone Thresholds, Batters",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Clean out any missing or NaN records in the coordinate metrics to prevent array collapse
clean_data = bat_challPitches.dropna(
    subset=["plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff", "hasABSChallenge"]
)

# Define exact, locked spatial grid ranges so both plots line up perfectly (in feet)
grid_range = [[-2.0, 2.0], [-2.0, 2.0]]
grid_bins = 30

# Compute 2D Binned Statistics (calculates the mean/rate of challenges per cell)
stat_top, xedges1, yedges1, binnumber1 = binned_statistic_2d(
    clean_data["plateTimeX"],
    clean_data["plateTimeY_topDiff"],
    values=clean_data["hasABSChallenge"],
    statistic="mean",
    bins=grid_bins,
    range=grid_range,
)

stat_bot, xedges2, yedges2, binnumber2 = binned_statistic_2d(
    clean_data["plateTimeX"],
    clean_data["plateTimeY_BotDiff"],
    values=clean_data["hasABSChallenge"],
    statistic="mean",
    bins=grid_bins,
    range=grid_range,
)

# Replace any completely empty cells (NaNs from division by zero) with 0 for plotting smoothness
stat_top = np.nan_to_num(stat_top)
stat_bot = np.nan_to_num(stat_bot)

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Top of the Strike Zone Horizon Rate ---
# We use imshow to plot the matrix, mapping it back to our true coordinate extent
im1 = axes[0].imshow(
    stat_top.T,
    origin="lower",
    extent=[-2.0, 2.0, -2.0, 2.0],
    cmap="viridis",
    aspect="auto",
    vmax=clean_data["hasABSChallenge"].mean() * 3, # Caps colorbar to highlight high-rate pockets
)
fig.colorbar(im1, ax=axes[0], label="Challenge Rate (Percentage of Challengeable Pitches)")

# Overlay theoretical boundaries (Home plate edges and the Y=0 boundary line)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=2.5, label="Top of Zone")
axes[0].axvline(x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges")
axes[0].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[0].set_title("Challenge Rate Relative to the Upper Zone Boundary")
axes[0].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[0].set_ylabel("Vertical Distance from Top Edge (Feet)")
axes[0].legend(loc="upper right")

# --- Plot 2: Bottom of the Strike Zone Horizon Rate ---
im2 = axes[1].imshow(
    stat_bot.T,
    origin="lower",
    extent=[-2.0, 2.0, -2.0, 2.0],
    cmap="viridis",
    aspect="auto",
    vmax=clean_data["hasABSChallenge"].mean() * 3,
)
fig.colorbar(im2, ax=axes[1], label="Challenge Rate (Percentage of Challengeable Pitches)")

# Overlay theoretical boundaries
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=2.5, label="Bottom of Zone")
axes[1].axvline(x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges")
axes[1].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[1].set_title("Challenge Rate Relative to the Lower Zone Boundary")
axes[1].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[1].set_ylabel("Vertical Distance from Bottom Edge (Feet)")
axes[1].legend(loc="upper right")

plt.suptitle(
    "Spatial Analysis of ABS Challenge Rates Against Physical Strike Zone Thresholds, Batters",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SPATIAL ANALYSIS: CHALLENGE RATES RELATIVE TO ZONE BOUNDARIES
# =============================================================================

# Define spatial binning grids to aggregate the challenge rates
# Adjust bin counts or ranges based on your data distribution if needed
x_bins = pd.cut(def_challPitches["plateTimeX"], bins=50)

# Bin the vertical differences
y_top_bins = pd.cut(def_challPitches["plateTimeY_topDiff"], bins=50)
y_bot_bins = pd.cut(def_challPitches["plateTimeY_BotDiff"], bins=50)

# Create Pivot Tables calculating the mean of "hasABSChallenge" (Challenge Rate)
heatmap_top = def_challPitches.pivot_table(
    index=y_top_bins,
    columns=x_bins,
    values="hasABSChallenge",
    aggfunc="mean",
    observed=False,
)

heatmap_bot = def_challPitches.pivot_table(
    index=y_bot_bins,
    columns=x_bins,
    values="hasABSChallenge",
    aggfunc="mean",
    observed=False,
)

# Invert Y-axis index format so higher pitches appear at the top of the heatmap visual
heatmap_top = heatmap_top.iloc[::-1]
heatmap_bot = heatmap_bot.iloc[::-1]

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap 1: Top of the Strike Zone Boundary Difference
sns.heatmap(
    heatmap_top,
    ax=axes[0],
    cmap="viridis",
    cbar_kws={"label": "Challenge Rate (Mean of hasABSChallenge)"},
)
axes[0].set_title("Challenge Rate Relative to the Top of the Strike Zone")
axes[0].set_ylabel("Vertical Distance from Top of Zone (plateTimeY_topDiff)")
axes[0].set_xlabel("Horizontal Plate Coordinate (plateTimeX)")

# Heatmap 2: Bottom of the Strike Zone Boundary Difference
sns.heatmap(
    heatmap_bot,
    ax=axes[1],
    cmap="viridis",
    cbar_kws={"label": "Challenge Rate (Mean of hasABSChallenge)"},
)
axes[1].set_title("Challenge Rate Relative to the Bottom of the Strike Zone")
axes[1].set_ylabel("Vertical Distance from Bottom of Zone (plateTimeY_BotDiff)")
axes[1].set_xlabel("Horizontal Plate Coordinate (plateTimeX)")

# Format label visibility across the large matrix grids
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle(
    "Spatial Analysis of ABS Challenge Probability Relative to Strike Zone Horizons, Defense",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SPATIAL ANALYSIS: TWO-DIMENSIONAL DISTRIBUTION WITH ZONE BOUNDARIES
# =============================================================================

# Isolate only rows where a challenge actually occurred
challenged_pitches = def_challPitches[def_challPitches["hasABSChallenge"] == 1]

# Explicitly drop any missing/NaN tracking records to prevent Numpy range errors
plot_data = challenged_pitches.dropna(
    subset=["plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff"]
)

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Top of the Strike Zone Horizon ---
counts1, xedges1, yedges1, im1 = axes[0].hist2d(
    plot_data["plateTimeX"],
    plot_data["plateTimeY_topDiff"],
    bins=25,
    cmap="viridis",
    cmin=1,  # Hides empty bins for visual clarity
)
fig.colorbar(im1, ax=axes[0], label="Volume of Challenges")

# Overlay theoretical boundaries (Home plate edges and the Y=0 boundary line)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=2, label="Top of Zone")
axes[0].axvline(
    x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges"
)
axes[0].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[0].set_title("Challenge Density Relative to the Upper Zone Boundary")
axes[0].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[0].set_ylabel("Vertical Distance from Top Edge (Feet)")
axes[0].legend(loc="upper right")
axes[0].set_xlim(-2.0, 2.0)  # Zoomed in closely around the plate area

# --- Plot 2: Bottom of the Strike Zone Horizon ---
counts2, xedges2, yedges2, im2 = axes[1].hist2d(
    plot_data["plateTimeX"],
    plot_data["plateTimeY_BotDiff"],  # Fixed a variable name typo here as well
    bins=25,
    cmap="viridis",
    cmin=1,
)
fig.colorbar(im2, ax=axes[1], label="Volume of Challenges")

# Overlay theoretical boundaries
axes[1].axhline(
    y=0, color="red", linestyle="--", linewidth=2, label="Bottom of Zone"
)
axes[1].axvline(
    x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges"
)
axes[1].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[1].set_title("Challenge Density Relative to the Lower Zone Boundary")
axes[1].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[1].set_ylabel("Vertical Distance from Bottom Edge (Feet)")
axes[1].legend(loc="upper right")
axes[1].set_xlim(-2.0, 2.0)

plt.suptitle(
    "Spatial Map of ABS Challenge Allocations Against Physical Strike Zone Thresholds, Defense",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Clean out any missing or NaN records in the coordinate metrics to prevent array collapse
clean_data = def_challPitches.dropna(
    subset=["plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff", "hasABSChallenge"]
)

# Define exact, locked spatial grid ranges so both plots line up perfectly (in feet)
grid_range = [[-2.0, 2.0], [-2.0, 2.0]]
grid_bins = 30

# Compute 2D Binned Statistics (calculates the mean/rate of challenges per cell)
stat_top, xedges1, yedges1, binnumber1 = binned_statistic_2d(
    clean_data["plateTimeX"],
    clean_data["plateTimeY_topDiff"],
    values=clean_data["hasABSChallenge"],
    statistic="mean",
    bins=grid_bins,
    range=grid_range,
)

stat_bot, xedges2, yedges2, binnumber2 = binned_statistic_2d(
    clean_data["plateTimeX"],
    clean_data["plateTimeY_BotDiff"],
    values=clean_data["hasABSChallenge"],
    statistic="mean",
    bins=grid_bins,
    range=grid_range,
)

# Replace any completely empty cells (NaNs from division by zero) with 0 for plotting smoothness
stat_top = np.nan_to_num(stat_top)
stat_bot = np.nan_to_num(stat_bot)

# Set up plotting area
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Top of the Strike Zone Horizon Rate ---
# We use imshow to plot the matrix, mapping it back to our true coordinate extent
im1 = axes[0].imshow(
    stat_top.T,
    origin="lower",
    extent=[-2.0, 2.0, -2.0, 2.0],
    cmap="viridis",
    aspect="auto",
    vmax=clean_data["hasABSChallenge"].mean() * 3, # Caps colorbar to highlight high-rate pockets
)
fig.colorbar(im1, ax=axes[0], label="Challenge Rate (Percentage of Challengeable Pitches)")

# Overlay theoretical boundaries (Home plate edges and the Y=0 boundary line)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=2.5, label="Top of Zone")
axes[0].axvline(x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges")
axes[0].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[0].set_title("Challenge Rate Relative to the Upper Zone Boundary")
axes[0].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[0].set_ylabel("Vertical Distance from Top Edge (Feet)")
axes[0].legend(loc="upper right")

# --- Plot 2: Bottom of the Strike Zone Horizon Rate ---
im2 = axes[1].imshow(
    stat_bot.T,
    origin="lower",
    extent=[-2.0, 2.0, -2.0, 2.0],
    cmap="viridis",
    aspect="auto",
    vmax=clean_data["hasABSChallenge"].mean() * 3,
)
fig.colorbar(im2, ax=axes[1], label="Challenge Rate (Percentage of Challengeable Pitches)")

# Overlay theoretical boundaries
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=2.5, label="Bottom of Zone")
axes[1].axvline(x=-0.708, color="white", linestyle="--", linewidth=2, label="Plate Edges")
axes[1].axvline(x=0.708, color="white", linestyle="--", linewidth=2)

axes[1].set_title("Challenge Rate Relative to the Lower Zone Boundary")
axes[1].set_xlabel("Horizontal Plate Coordinate (Feet)")
axes[1].set_ylabel("Vertical Distance from Bottom Edge (Feet)")
axes[1].legend(loc="upper right")

plt.suptitle(
    "Spatial Analysis of ABS Challenge Rates Against Physical Strike Zone Thresholds, Defense",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

## Box plots of umpire challenge and overturn rates

In [ ]:
# =============================================================================
# UMPIRE PERFORMANCE ANALYSIS: CHALLENGE & OVERTURN RATES
# =============================================================================

# --- Process Batter Challenges by Umpire ---
ump_bat_group = bat_challPitches.groupby("homePlateUmpireId").agg(
    Total_Opps=("homePlateUmpireId", "size"),
    Total_Challenges=("hasABSChallenge", "sum"),
    Total_Overturned=("isOverturned", "sum"),
).reset_index()

# Calculate rates
ump_bat_group["Challenge_Rate"] = (
    ump_bat_group["Total_Challenges"] / ump_bat_group["Total_Opps"]
)
ump_bat_group["Overturn_Rate"] = (
    ump_bat_group["Total_Overturned"] / ump_bat_group["Total_Challenges"]
)
ump_bat_group["Side"] = "Batter Split"

# Process Defensive Challenges by Umpire ---
ump_def_group = def_challPitches.groupby("homePlateUmpireId").agg(
    Total_Opps=("homePlateUmpireId", "size"),
    Total_Challenges=("hasABSChallenge", "sum"),
    Total_Overturned=("isOverturned", "sum"),
).reset_index()

# Calculate rates
ump_def_group["Challenge_Rate"] = (
    ump_def_group["Total_Challenges"] / ump_def_group["Total_Opps"]
)
ump_def_group["Overturn_Rate"] = (
    ump_def_group["Total_Overturned"] / ump_def_group["Total_Challenges"]
)
ump_def_group["Side"] = "Defense Split"

# Filter for Minimum Opportunities and Combine Data ---
# Filters out low sample sizes to prevent skewing the distributions
min_opps = 100
ump_bat_filtered = ump_bat_group[ump_bat_group["Total_Opps"] >= min_opps]
ump_def_filtered = ump_def_group[ump_def_group["Total_Opps"] >= min_opps]

combined_ump_df = pd.concat([ump_bat_filtered, ump_def_filtered], axis=0)

# --- 4. Plotting Side-by-Side Boxplots ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxplot 1: Challenge Rates by Split
sns.boxplot(
    x="Side",
    y="Challenge_Rate",
    data=combined_ump_df,
    ax=axes[0],
    palette="Pastel1",
    hue="Side",
    legend=False,
)
axes[0].set_title("Umpire Challenge Rates by Matchup Perspective")
axes[0].set_ylabel("Challenge Rate")
axes[0].set_xlabel("Dataset Split")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Boxplot 2: Overturn Rates by Split
sns.boxplot(
    x="Side",
    y="Overturn_Rate",
    data=combined_ump_df,
    ax=axes[1],
    palette="Pastel2",
    hue="Side",
    legend=False,
)
axes[1].set_title("Umpire Overturn Rates by Matchup Perspective")
axes[1].set_ylabel("Overturn Rate")
axes[1].set_xlabel("Dataset Split")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle(
    f"Umpire Variability Analysis Across ABS Challenge Splits (Minimum {min_opps} Opportunities)",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

# =============================================================================
# STATISTICAL OUTLIER IDENTIFICATION (UMPIRES)
# =============================================================================

metrics_to_check = ["Challenge_Rate", "Overturn_Rate"]
sides_to_check = ["Batter Split", "Defense Split"]

print("\n--- DETECTED UMPIRE OUTLIERS BY perspectives ---")

for side in sides_to_check:
    # Filter by dataset split context
    df_side = combined_ump_df[combined_ump_df["Side"] == side]
    print(f"\n===== Split: {side} =====")

    for metric in metrics_to_check:
        # Calculate IQR parameters explicitly for this split
        q1 = df_side[metric].quantile(0.25)
        q3 = df_side[metric].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Isolate outlier rows
        outliers_df = df_side[(df_side[metric] < lower_bound) | (df_side[metric] > upper_bound)]

        print(f"-> Metric: {metric} (Bounds: {lower_bound:.4f} to {upper_bound:.4f})")
        if not outliers_df.empty:
            print(f"   Found {len(outliers_df)} anomalous umpire record(s):")
            # Display core identifiers for your EDA write-up
            print(outliers_df[["homePlateUmpireId", "Total_Opps", "Total_Challenges", metric]].to_string(index=False))
        else:
            print("   ✅ No statistical outliers detected for this metric channel.")

## Time Series for Challenge and Overturn rates

In [ ]:
# Import schedule dataset
df_sched = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-Schedule.csv", encoding="latin1")
df_teams = pd.read_csv("https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-Teams.csv", encoding="latin1")

In [ ]:
# =============================================================================
# TIME SERIES ANALYSIS: MONTHLY & DAILY ABS SPLITS
# =============================================================================

# Merge Temporal Features from the Schedule DataFrame ---
# Isolate only the unique game key and date markers to avoid column inflation
date_lookup = df_sched[["gamePk", "gameMonth", "gameDay"]].drop_duplicates()

bat_time_merged = pd.merge(bat_challPitches, date_lookup, on="gamePk", how="inner")
def_time_merged = pd.merge(def_challPitches, date_lookup, on="gamePk", how="inner")


# Monthly Aggregations (Batters vs. Defense) ---
bat_monthly = (
    bat_time_merged.groupby("gameMonth")
    .agg(
        Total_Pitches=("hasABSChallenge", "count"),
        Total_Challenges=("hasABSChallenge", "sum"),
        Total_Overturned=("isOverturned", "sum"),
    )
    .reset_index()
)
bat_monthly["Challenge_Rate"] = (
    bat_monthly["Total_Challenges"] / bat_monthly["Total_Pitches"]
)
bat_monthly["Overturn_Rate"] = (
    bat_monthly["Total_Overturned"] / bat_monthly["Total_Challenges"]
)
bat_monthly["Side"] = "Batter"

def_monthly = (
    def_time_merged.groupby("gameMonth")
    .agg(
        Total_Pitches=("hasABSChallenge", "count"),
        Total_Challenges=("hasABSChallenge", "sum"),
        Total_Overturned=("isOverturned", "sum"),
    )
    .reset_index()
)
def_monthly["Challenge_Rate"] = (
    def_monthly["Total_Challenges"] / def_monthly["Total_Pitches"]
)
def_monthly["Overturn_Rate"] = (
    def_monthly["Total_Overturned"] / def_monthly["Total_Challenges"]
)
def_monthly["Side"] = "Defense"


# Daily Aggregations (Constructing Continuous Datetime Timelines) ---
bat_daily = (
    bat_time_merged.groupby(["gameMonth", "gameDay"])
    .agg(
        Total_Pitches=("hasABSChallenge", "count"),
        Total_Challenges=("hasABSChallenge", "sum"),
        Total_Overturned=("isOverturned", "sum"),
    )
    .reset_index()
)

# Convert to continuous datetime index (Assuming current season 2026)
bat_daily["Date"] = pd.to_datetime(
    {
        "year": 2026,
        "month": bat_daily["gameMonth"],
        "day": bat_daily["gameDay"],
    }
)
bat_daily["Challenge_Rate"] = (
    bat_daily["Total_Challenges"] / bat_daily["Total_Pitches"]
)
bat_daily["Overturn_Rate"] = (
    bat_daily["Total_Overturned"] / bat_daily["Total_Challenges"]
)

def_daily = (
    def_time_merged.groupby(["gameMonth", "gameDay"])
    .agg(
        Total_Pitches=("hasABSChallenge", "count"),
        Total_Challenges=("hasABSChallenge", "sum"),
        Total_Overturned=("isOverturned", "sum"),
    )
    .reset_index()
)
def_daily["Date"] = pd.to_datetime(
    {
        "year": 2026,
        "month": def_daily["gameMonth"],
        "day": def_daily["gameDay"],
    }
)
def_daily["Challenge_Rate"] = (
    def_daily["Total_Challenges"] / def_daily["Total_Pitches"]
)
def_daily["Overturn_Rate"] = (
    def_daily["Total_Overturned"] / def_daily["Total_Challenges"]
)


# Set up plotting area
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Define the explicit datetime boundaries for the All-Star Break
asb_start = pd.to_datetime("2026-07-13")
asb_end = pd.to_datetime("2026-07-16")

# Plot 1: Monthly Challenge Rates
axes[0, 0].plot(
    bat_monthly["gameMonth"],
    bat_monthly["Challenge_Rate"],
    marker="o",
    color="teal",
    label="Batters",
)
axes[0, 0].plot(
    def_monthly["gameMonth"],
    def_monthly["Challenge_Rate"],
    marker="s",
    color="coral",
    label="Defense",
)
axes[0, 0].set_title("Monthly Challenge Rates")
axes[0, 0].set_xlabel("Month")
axes[0, 0].set_ylabel("Rate")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

# Plot 2: Monthly Overturn Rates
axes[0, 1].plot(
    bat_monthly["gameMonth"],
    bat_monthly["Overturn_Rate"],
    marker="o",
    color="teal",
    label="Batters",
)
axes[0, 1].plot(
    def_monthly["gameMonth"],
    def_monthly["Overturn_Rate"],
    marker="s",
    color="coral",
    label="Defense",
)
axes[0, 1].set_title("Monthly Overturn Rates")
axes[0, 1].set_xlabel("Month")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# Plot 3: Daily Challenge Rate Timeline
axes[1, 0].plot(
    bat_daily["Date"],
    bat_daily["Challenge_Rate"],
    color="teal",
    alpha=0.6,
    label="Batters",
)
axes[1, 0].plot(
    def_daily["Date"],
    def_daily["Challenge_Rate"],
    color="coral",
    alpha=0.6,
    label="Defense",
)
# Add vertical shaded block for All-Star Break
axes[1, 0].axvspan(
    asb_start, asb_end, color="gray", alpha=0.3, label="All-Star Break"
)
axes[1, 0].text(
    pd.to_datetime("2026-07-14"),
    axes[1, 0].get_ylim()[1] * 0.85,
    "ASB",
    color="black",
    weight="bold",
    ha="center",
    fontsize=9,
)
axes[1, 0].set_title("Daily Challenge Rates")
axes[1, 0].set_xlabel("Timeline")
axes[1, 0].set_ylabel("Rate")
axes[1, 0].legend()
axes[1, 0].tick_params(axis="x", rotation=30)
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# Plot 4: Daily Overturn Rate Timeline
axes[1, 1].plot(
    bat_daily["Date"],
    bat_daily["Overturn_Rate"],
    color="teal",
    alpha=0.6,
    label="Batters",
)
axes[1, 1].plot(
    def_daily["Date"],
    def_daily["Overturn_Rate"],
    color="coral",
    alpha=0.6,
    label="Defense",
)
# Add vertical shaded block for All-Star Break
axes[1, 1].axvspan(
    asb_start, asb_end, color="gray", alpha=0.3, label="All-Star Break"
)
axes[1, 1].text(
    pd.to_datetime("2026-07-14"),
    axes[1, 1].get_ylim()[1] * 0.85,
    "ASB",
    color="black",
    weight="bold",
    ha="center",
    fontsize=9,
)
axes[1, 1].set_title("Daily Overturn Rates")
axes[1, 1].set_xlabel("Timeline")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()
axes[1, 1].tick_params(axis="x", rotation=30)
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    "Temporal ABS Dynamics: Challenge and Overturn Rates Across the 2026 Season",
    fontsize=14,
    weight="bold",
)
plt.tight_layout()
plt.show()

## Team scatterplots

In [ ]:
# Batting data assignments
bat_challPitches["Batting_Team_ID"] = np.where(bat_challPitches["inningHalf"] == 0,
                                               bat_challPitches["awayTeamId"],
                                               bat_challPitches["homeTeamId"])

# Defensive data assignments
def_challPitches["Defending_Team_ID"] = np.where(def_challPitches["inningHalf"] == 0,
                                                 def_challPitches["homeTeamId"],
                                                 def_challPitches["awayTeamId"])

# --- CALCULATE AGGREGATE TEAM METRICS ---
# Offensive metrics (from batter-challengeable situations)
bat_stats = bat_challPitches.groupby("Batting_Team_ID").agg(
    total_strike_situations=("hasABSChallenge", "count"),
    total_bat_challenges=("hasABSChallenge", "sum"),
    overturned_bat_challenges=("isOverturned", "sum")
).reset_index()

bat_stats["Batter_Challenge_Rate"] = bat_stats["total_bat_challenges"] / bat_stats["total_strike_situations"]
bat_stats["Batter_Overturn_Rate"] = bat_stats["overturned_bat_challenges"] / bat_stats["total_bat_challenges"]

# Defensive metrics (from defensive-challengeable situations)
def_stats = def_challPitches.groupby("Defending_Team_ID").agg(
    total_ball_situations=("hasABSChallenge", "count"),
    total_def_challenges=("hasABSChallenge", "sum"),
    overturned_def_challenges=("isOverturned", "sum")
).reset_index()

def_stats["Defense_Challenge_Rate"] = def_stats["total_def_challenges"] / def_stats["total_ball_situations"]
def_stats["Defense_Overturn_Rate"] = def_stats["overturned_def_challenges"] / def_stats["total_def_challenges"]

# --- JOIN WITH METADATA & MERGE MAPS ---
# Merge batting and defensive records together
team_matrix = pd.merge(bat_stats, def_stats, left_on="Batting_Team_ID", right_on="Defending_Team_ID")

# Join with df_teams lookup to map names
team_matrix = pd.merge(team_matrix, df_teams[["id", "name", "teamCode"]], left_on="Batting_Team_ID", right_on="id")
#team_matrix.rename(columns={"teamCode": "Team"}, inplace=True)

# --- GENERATE THE 2X2 GRID OF SCATTER PLOTS ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Team-Level Challenge and Overturn Rate Comparisons", fontsize=16, weight="bold")

# Plot 1: Challenge vs Overturn (Batters)
sns.scatterplot(ax=axes[0, 0], data=team_matrix, x="Batter_Challenge_Rate", y="Batter_Overturn_Rate", s=100, color="blue")
axes[0, 0].set_title("Offensive Challenge Rate vs. Overturn Rate")

# Plot 2: Challenge vs Overturn (Defense)
sns.scatterplot(ax=axes[0, 1], data=team_matrix, x="Defense_Challenge_Rate", y="Defense_Overturn_Rate", s=100, color="orange")
axes[0, 1].set_title("Defensive Challenge Rate vs. Overturn Rate")

# Plot 3: Overturn Comparison (Batters vs Defense)
sns.scatterplot(ax=axes[1, 0], data=team_matrix, x="Batter_Overturn_Rate", y="Defense_Overturn_Rate", s=100, color="green")
axes[1, 0].set_title("Team Overturn Rate Comparison, Offensive vs Defensive")

# Plot 4: Challenge Comparison (Batters vs Defense)
sns.scatterplot(ax=axes[1, 1], data=team_matrix, x="Batter_Challenge_Rate", y="Defense_Challenge_Rate", s=100, color="purple")
axes[1, 1].set_title("Team Challenge Rate Comparison, Offensive vs Defensive")

# Structural annotation loop for adding text abbreviations to axes grid
for ax, x_col, y_col in [
    (axes[0, 0], "Batter_Challenge_Rate", "Batter_Overturn_Rate"),
    (axes[0, 1], "Defense_Challenge_Rate", "Defense_Overturn_Rate"),
    (axes[1, 0], "Batter_Overturn_Rate", "Defense_Overturn_Rate"),
    (axes[1, 1], "Batter_Challenge_Rate", "Defense_Challenge_Rate")
]:
    x_vals = team_matrix[x_col].values
    y_vals = team_matrix[y_col].values

    # Compute and plot the Linear Trend Line
    if len(x_vals) > 1:
        # Fit a 1st-degree polynomial (y = mx + b)
        slope, intercept = np.polyfit(x_vals, y_vals, 1)
        x_trend = np.linspace(x_vals.min(), x_vals.max(), 100)
        y_trend = slope * x_trend + intercept

        # Plot the trend line as a dashed line matching the scatter color palette
        ax.plot(x_trend, y_trend, color="red", linestyle="--", lw=2, alpha=0.8,
                label=f"Trend (m={slope:.2f})")
        ax.legend(loc="best")

    texts = []
    for idx, row in team_matrix.iterrows():
        # Place text directly on the point initially
        t = ax.text(row[x_col], row[y_col], row["teamCode"], fontsize=9, weight="bold")
        texts.append(t)

    # Repel overlapping text labels from each other and from the points
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle="-", color="gray", alpha=0.5, lw=0.5), # Adds clean mini-pointer lines if text moves too far
        expand_points=(1.5, 1.5),
        force_points=0.1
    )

plt.tight_layout()
plt.show()